In [1]:
!pip install -r HurrReqs.txt

Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cu128
  Using cached shapely-2.1.2-cp314-cp314-win_amd64.whl.metadata (7.1 kB)
Using cached geopandas-1.1.3-py3-none-any.whl (342 kB)
   ---------------------------------------- 0.0/23.6 MB ? eta -:--:--
   ----------------- ---------------------- 10.2/23.6 MB 54.6 MB/s eta 0:00:01
   ---------------------------------------- 23.6/23.6 MB 63.5 MB/s  0:00:00
   ---------------------------------------- 0.0/6.4 MB ? eta -:--:--
   ---------------------------------------- 6.4/6.4 MB 75.0 MB/s  0:00:00
Using cached shapely-2.1.2-cp314-cp314-win_amd64.whl (1.8 MB)

   ---------------------------------------- 0/4 [shapely]
   ---------------------------------------- 0/4 [shapely]
   ---------------------------------------- 0/4 [shapely]
   ---------------------------------------- 0/4 [shapely]
   ---------- ----------------------------- 1/4 [pyproj]
   ---------- ----------------------------- 1/4 [pyproj]
   --------

In [2]:
import pandas as pd
import numpy as np
import geopandas as gpd
import glob
import os
from shapely.geometry import Point


In [5]:
States = ['FL', 'TX', 'LA', 'MS', 'AL', 'GA', 'SC', 'NC',
    'VA', 'MD', 'DE', 'NJ', 'NY', 'CT', 'RI', 'MA',
    'ME', 'NH', 'PR', 'VI']

print("loading NFIP data...")
nfip_df = pd.read_csv(r'fimaclaims.csv', 
                      dtype={
                           'state':               'str',
                            'dateOfLoss':          'str',
                            'floodEvent':          'str',
                            'causeOfDamage':       'str',
                            'ratedFloodZone':      'str',
                            'countyCode':          'str',
                            'censusTract':         'str',
                            'censusBlockGroupFips':'str',
                            'occupancyType':       'str',
                            'floodZoneCurrent':    'str',
                      },
                    low_memory=False                    
)
print(f'rows: {nfip_df.shape[0]}, columns: {nfip_df.shape[1]}')



loading NFIP data...
rows: 2719462, columns: 73


In [7]:
nfip_df = nfip_df[nfip_df['state'] != 'UN'].copy()
print(f"dropped unknown states, {len(nfip_df)} rows remaining")


nfip_df = nfip_df[nfip_df['state'].isin(States)].copy()
print(f"filtered to coastal states, {len(nfip_df)} rows remaining")

nfip_df = nfip_df.dropna(subset=['latitude', 'longitude'])
print(f'dropped rows with missing latitude or longitude, {len(nfip_df)} rows remaining')

nfip_df['dateOfLoss'] = pd.to_datetime(nfip_df['dateOfLoss'], format='mixed',dayfirst=False)
print(f"\ndateOfLoss range: {nfip_df['dateOfLoss'].min()} -> {nfip_df['dateOfLoss'].max()}")

nfip_df = nfip_df[nfip_df['dateOfLoss'].dt.year >= 1996].copy()
print(f"filtered to 1996 and later, {len(nfip_df)} rows remaining")

hurricane_keywords = ['hurricane', 'tropical storm', 'tropical depression']
pattern = '|'.join(hurricane_keywords)

nfip_df = nfip_df[nfip_df['floodEvent'].str.contains(pattern, case=False, na=False)].copy()
print(f"filtered to hurricane-related events, {len(nfip_df)} rows remaining")

nfip_df = nfip_df[
    nfip_df['buildingDamageAmount'].notna() & (nfip_df['buildingDamageAmount'] > 0)].copy()

print(f"After damage over 0: {len(nfip_df)} rows remaining")

print("\nSample of cleaned NFIP data:")
nfip_df.head(10)

dropped unknown states, 910075 rows remaining
filtered to coastal states, 910075 rows remaining
dropped rows with missing latitude or longitude, 910075 rows remaining

dateOfLoss range: 1996-07-12 00:00:00+00:00 -> 2024-11-11 00:00:00+00:00
filtered to 1996 and later, 910075 rows remaining
filtered to hurricane-related events, 910075 rows remaining
After damage over 0: 910075 rows remaining

Sample of cleaned NFIP data:


,agricultureStructureIndicator,asOfDate,basementEnclosureCrawlspaceType,policyCount,crsClassificationCode,dateOfLoss,elevatedBuildingIndicator,elevationCertificateIndicator,elevationDifference,baseFloodElevation,...,rentalPropertyIndicator,state,reportedCity,reportedZipCode,countyCode,censusTract,censusBlockGroupFips,latitude,longitude,id
0,0,2026-01-12T00:00:00.000Z,2.0,1,NaN,2008-09-12 00:00:00+00:00,1,NaN,NaN,NaN,...,0,TX,Currently Unavailable,77650,48167,48167723900,481677239003,29.4,-94.8,4ae4e90e-d6e5-44c3-8b77-09aea5408e13
1,0,2026-01-12T00:00:00.000Z,NaN,1,NaN,2017-08-27 00:00:00+00:00,0,1,NaN,NaN,...,1,TX,Currently Unavailable,77478,48157,48157671601,481576716014,29.6,-95.6,1d64a17f-e908-4016-9936-c6ff3b115867
3,0,2026-01-12T00:00:00.000Z,NaN,1,NaN,2008-09-13 00:00:00+00:00,0,NaN,NaN,NaN,...,0,TX,Currently Unavailable,77037,48201,48201222401,482012224012,29.9,-95.4,75dcdbbf-e3f4-4ea7-a8e5-42fcdc96aeb7
6,0,2026-01-12T00:00:00.000Z,NaN,1,7.0,2012-10-30 00:00:00+00:00,0,NaN,NaN,NaN,...,0,NJ,Currently Unavailable,07734,34025,34025801700,340258017003,40.4,-74.1,33e7e663-c06d-4aa0-9c32-faec151e64cf
7,0,2026-01-12T00:00:00.000Z,NaN,1,NaN,2008-09-13 00:00:00+00:00,1,NaN,6.0,12.0,...,0,TX,Currently Unavailable,77568,48167,48167723800,481677238001,29.3,-94.9,eac519b6-193a-4ed0-b59f-19d2655d12f1
19,0,2026-01-12T00:00:00.000Z,2.0,1,NaN,2011-08-27 00:00:00+00:00,0,1,NaN,NaN,...,0,NJ,Currently Unavailable,08846,34023,34023000200,340230002004,40.6,-74.5,06c6d85d-ea61-487e-ae57-4e8b753318fe
33,0,2026-01-12T00:00:00.000Z,NaN,1,NaN,2016-09-02 00:00:00+00:00,0,NaN,NaN,NaN,...,0,FL,Currently Unavailable,34448,12017,12017451700,120174517002,28.8,-82.6,d77978d7-eaf0-4077-9ebf-52e1949c5950
34,0,2026-01-12T00:00:00.000Z,NaN,1,NaN,2005-08-29 00:00:00+00:00,0,NaN,NaN,NaN,...,0,LA,Currently Unavailable,70122,22071,22071013800,220710138003,30.0,-90.1,a35f4f15-2cf4-4a27-9df1-85d384c9d7d4
36,0,2026-01-12T00:00:00.000Z,0.0,1,NaN,2017-08-26 00:00:00+00:00,0,NaN,NaN,NaN,...,0,TX,Currently Unavailable,77089,48201,48201350300,482013503001,29.6,-95.2,1056485a-03d4-4967-bc99-f42692748a9a
37,0,2026-01-12T00:00:00.000Z,NaN,1,NaN,2011-08-28 00:00:00+00:00,1,NaN,2.0,14.0,...,0,NY,Currently Unavailable,11978,36103,36103190504,361031905041,40.8,-72.7,fff86222-b276-4973-97ff-79088fdc0b3a
